In [8]:
import torch_directml
import sys


In [10]:
pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install ipywidgets


   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 322.4 kB/s eta 0:00:06
   --------- ------------------------------ 0.5/2.2 MB 322.4 kB/s eta 0:00:06
   --------- ------------------------------ 0.5/2.2 MB 322.4 kB/s eta 0:00:06
   --------- ------------------------------ 0.5/2.2 MB 322.4 kB/s eta 0:00:06
   -------------- ------------------------- 0.8/2.2 MB 319.5 kB/s eta 0:00:05
   -------------- ------------------------- 0.8/2.2 MB 319

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('HF_TOKEN')

## Dataset Collection

In [13]:
pip install huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [14]:
from pathlib import Path

DATA_DIR = Path("data/plantvillage").resolve()

CLASS_MAP = {
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "maize_gray_leaf_spot",
    "Corn_(maize)___Common_rust_": "maize_common_rust",
    "Corn_(maize)___Northern_Leaf_Blight": "maize_northern_leaf_blight",
    "Corn_(maize)___healthy": "maize_healthy",
    "Tomato___Bacterial_spot": "tomato_bacterial_spot",
    "Tomato___Early_blight": "tomato_early_blight",
    "Tomato___Late_blight": "tomato_late_blight",
    "Tomato___Leaf_Mold": "tomato_leaf_mold",
    "Tomato___healthy": "tomato_healthy",
}

print("Download location:", DATA_DIR)
print("Classes:", len(CLASS_MAP))

Download location: C:\Users\baabyface\Documents\my work\agrilens\agrilens-training\data\plantvillage
Classes: 9


In [15]:
from huggingface_hub import HfApi, hf_hub_download
from pathlib import Path
from zipfile import ZipFile
import shutil


In [18]:

repo_id = "mohanty/PlantVillage"
revision = HfApi().dataset_info(repo_id).sha

# Downloads the archive; Hugging Face caches it locally.
archive_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    revision=revision,
    filename="data.zip",
)


In [19]:

# Inspect archive paths before extracting anything.
with ZipFile(archive_path) as archive:
    matches = {}

    for folder in CLASS_MAP:
        marker = f"/color/{folder}/"
        matches[folder] = [
            name for name in archive.namelist()
            if marker in "/" + name.replace("\\", "/")
            and Path(name).suffix.lower() in {".jpg", ".jpeg", ".png"}
        ]

    for folder, names in matches.items():
        print(f"{CLASS_MAP[folder]:32} {len(names):5} images")

    missing = [folder for folder, names in matches.items() if not names]
    if missing:
        print("Example archive paths:", archive.namelist()[:15])
        raise RuntimeError(
            f"Missing classes: {missing}. Stop here and share the output."
        )

    # Extract only our nine colour-image classes.
    for folder, names in matches.items():
        destination = DATA_DIR / "raw" / "color" / folder
        destination.mkdir(parents=True, exist_ok=True)

        for name in names:
            target = destination / Path(name).name
            with archive.open(name) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)



maize_gray_leaf_spot               513 images
maize_common_rust                 1192 images
maize_northern_leaf_blight         985 images
maize_healthy                     1162 images
tomato_bacterial_spot             2127 images
tomato_early_blight               1000 images
tomato_late_blight                1909 images
tomato_leaf_mold                   952 images
tomato_healthy                    1591 images


In [20]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / "source_revision.txt").write_text(
    f"{repo_id}\n{revision}\n",
    encoding="utf-8",
)

print("Extraction complete.")

Extraction complete.
